In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

import MCTDHF_jax
from electron_integrals import *

from CI_physicist import *

In [ ]:
import importlib
importlib.reload(MCTDHF_jax)

Define number of electrons and orbitals

In [ ]:
# Number of orbitals (without spin)
num_orbitals = 10
# Number of electrons
num_electrons = 2
# Number of MCTDHF orbitals
num_mctdhf_orbitals = 5
#Include spin?
include_spin = False
spin_factor = 1+int(include_spin)

num_spin_orbitals = spin_factor*num_orbitals

Calculate electron integrals

In [ ]:
x_max = 10
num_points = 1000

x = np.linspace(-x_max,x_max,num_points)

#pot = GaussianWell(w=100, a=1, center=0)
pot = HOPotential()

spf, h = get_spf_and_diag_h(num_orbitals, x, pot)

if include_spin:
    #Add spin
    h = np.kron(h, np.eye(2,2))

g = coulomb_interaction_matrix_elements(spf, spf, x, x, kappa = 1, a = 0.01)

if include_spin:
    #Add spin
    g = np.kron(g, np.einsum("pr,qs->pqrs",np.eye(2,2), np.eye(2,2)))

print('Sanity test, due to symmetry in g this should be zero:')
print(-g[2,1,1,0]+g[2,0,1,1]+g[1,1,2,0]-g[1,0,2,1])

In [ ]:
mctdhf = MCTDHF_jax.MCTDHF(jnp.array(h, jnp.complex64), jnp.array(g, jnp.complex64), num_mctdhf_orbitals, num_spin_orbitals, num_electrons)
mctdhf.run_imag_time_prop(jnp.array(0, jnp.complex64), jnp.array(1, jnp.complex64), jnp.array(0.1, jnp.complex64))

In [ ]:
mctdhf_slater_dets = np.array(CIHamiltonian.get_slater_dets(num_mctdhf_orbitals, num_electrons))

def get_RDMs(C):
        C_conj = C.conj()

        D = jnp.zeros((num_mctdhf_orbitals,)*2 , dtype=jnp.complex64)
        d = jnp.zeros((num_mctdhf_orbitals,)*4, dtype=jnp.complex64)

        for n, det_n in enumerate(mctdhf_slater_dets):
            for m, det_m in enumerate(mctdhf_slater_dets):
                num_differences = np.sum(np.abs(det_n-det_m))
                match(num_differences):
                    case 0:
                        for p, n_p in enumerate(det_n):
                            D.at[p,p].add(n_p*C_conj[m]*C[n])
                            for r, n_r in enumerate(det_n):
                                d.at[p,r,p,r].add(n_p*n_r*C_conj[m]*C[n])
                                d.at[p,r,r,p].add(-n_p*n_r*C_conj[m]*C[n]) # Note the sign                 
                    case 2:
                        p = np.flatnonzero(np.asarray((det_m-det_n)==1))[0]
                        q = np.flatnonzero(np.asarray((det_n-det_m)==1))[0]
                        gamma = (-1)**(np.sum(det_n[:q])+np.sum(det_m[:p]))
                        D.at[p,q].add(gamma*C_conj[m]*C[n])

                        for r, n_r in enumerate(det_n):
                            val = gamma*n_r*C_conj[m]*C[n]
                            d.at[p,r,q,r].add(val)
                            d.at[r,p,q,r].add(-val) # Note the sign
                            d.at[p,r,r,q].add(-val) # Note the sign
                            d.at[r,p,r,q].add(val)

                    case 4:
                        p,q = np.flatnonzero(np.asarray((det_m-det_n)==1))
                        r,s = np.flatnonzero(np.asarray((det_n-det_m)==1))
                        
                        gamma = np.sum(det_n[:r])
                        gamma += np.sum(det_n[:s])-1 # -1 since r<s 
                        gamma += np.sum(det_n[:q])-int(s<q)-int(r<q)
                        gamma += np.sum(det_n[:p])-int(s<p)-int(r<p) # No additional term since p<q

                        val = C_conj[m]*C[n]*(-1)**gamma
                        d.at[p,q,r,s].add(val)
                        d.at[q,p,r,s].add(-val) # Note the sign
                        d.at[p,q,s,r].add(-val) # Note the sign
                        d.at[q,p,s,r].add(val)

        return D, d

In [ ]:
from scipy.special import comb

In [ ]:
num_slater_dets = comb(num_mctdhf_orbitals, num_electrons, exact = True)

C = rng.random(num_slater_dets).astype(np.cdouble)
# Normalize
C = np.divide(C, np.sqrt(C.T.conj()@C))

D = get_RDMs(C)

In [ ]:
h = jnp.array(h, jnp.complex64)
g = jnp.array(g, jnp.complex64)

rng = np.random.default_rng()
r = rng.random((num_spin_orbitals, num_mctdhf_orbitals))
u, _, vh = np.linalg.svd(r, full_matrices=False)
b = (u@vh).astype(np.cdouble)
bc = b.conj()

h_1 = jnp.einsum('jm, ij -> im', b, h)
h_2 = jnp.einsum('in, im -> nm', bc, h_1)
h_3 = jnp.einsum('in, nm -> im', b, h_2)

g_2 = jnp.einsum('jq, ls, ijkl -> iqks', bc, b, g)
g_3 = jnp.einsum('kr, iqks -> iqrs', b, g_2)
g_4 = jnp.einsum('ip, iqrs -> pqrs', bc, g_3)
g_5 = jnp.einsum('ip, pqrs -> iqrs', b, g_4)


SlaterCondonHamiltonian(num_mctdhf_orbitals, num_electrons, h_2, g_4).get_hamiltonian()

In [ ]:
from jax import jit

@jit
def imag_time_int(C, b, h, g):
    bc = b.conj()

    h_1 = jnp.einsum('jm, ij -> im', b, h)
    h_2 = jnp.einsum('in, im -> nm', bc, h_1)
    h_3 = jnp.einsum('in, nm -> im', b, h_2)

    g_2 = jnp.einsum('jq, ls, ijkl -> iqks', bc, b, g)
    g_3 = jnp.einsum('kr, iqks -> iqrs', b, g_2)
    g_4 = jnp.einsum('ip, iqrs -> pqrs', bc, g_3)
    g_5 = jnp.einsum('ip, pqrs -> iqrs', b, g_4)

    
    # D, d = self.get_RDMs(C)
    # D_inv = jnp.linalg.pinv(D)

    # b_dot = -(h_1 - h_3 + jnp.einsum('np, pqrs, iqrs -> in', D_inv, d, g_3-g_5))


    H = SlaterCondonHamiltonian(num_mctdhf_orbitals, num_electrons, h_2, g_4).get_hamiltonian()

    ## From Beck paper, replacing HC with (H-IE)C should keep the wave function normalized
    # E = C.conj().T@H@C/(C.conj().T@C)
    # C_dot = - (H-self.I*E)@C 

    return 0 #self.Cb_to_y(C_dot, b_dot)

In [ ]:
imag_time_int(C, b, h, g)